# 🎓 考卷手寫筆跡分離模型微調 (在線合成 On-The-Fly 訓練版)

本筆記本採用**在線邊訓練邊合成 (On-the-fly Synthesis)** 架構：
- **100% 真實手寫掃描素材**：42 頁整頁草稿層 + 391 個真實手寫記號，絕無程式繪製偽造線條。
- **輕量資產包**：只需上傳 `exam_assets_lightweight.zip`（僅約 47 MB），告別數百 MB 的巨大圖片上傳。
- **資料多樣性無限**：每個 Batch 即時在乾淨試卷上隨機填答、圈選選項、壓線塗改與草稿貼附，永遠不會過擬合。
- **模型架構**：ResNet34-UNet (BCE + Dice Loss 複合損失)。
- **預估時間**：在 Colab 免費 T4 GPU 上約 10 ~ 15 分鐘。

In [ ]:
# 1. 檢查 GPU 環境並安裝依賴套件
!nvidia-smi
!pip install -q pymupdf segmentation-models-pytorch opencv-python-headless


In [ ]:
# 2. 載入輕量資產包 (exam_assets_lightweight.zip - 僅約 47 MB)
import os, zipfile

zip_path = None
if os.path.exists('/content/drive/MyDrive/exam_assets_lightweight.zip'):
    zip_path = '/content/drive/MyDrive/exam_assets_lightweight.zip'
elif os.path.exists('exam_assets_lightweight.zip'):
    zip_path = 'exam_assets_lightweight.zip'
else:
    from google.colab import files
    print('請上傳本地的 exam_assets_lightweight.zip (約 47 MB):')
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]

print(f'正在解壓縮輕量資產包 {zip_path}...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('.')

print('🎉 資產解壓完成！目錄狀態：')
!ls -la assets clean_pdfs


In [ ]:
# 3. 一鍵啟動在線合成訓練 (On-The-Fly Training)
# 預設設定：25 輪，每輪 200 步隨機合成樣本，批次大小 4，解析度 1024x768
!python tools/train_on_the_fly.py \
    --epochs 25 \
    --batch_size 4 \
    --steps_per_epoch 200 \
    --val_steps 30 \
    --height 1024 \
    --width 768 \
    --save_path weights/best_exam_remover.pth


In [ ]:
# 4. 驗證與可視化檢測：隨機合成並檢視去手寫效果
import torch, cv2
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
from synthetic_engine_v2 import ExamSynthesizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=1).to(device)
model.load_state_dict(torch.load('weights/best_exam_remover.pth', map_location=device))
model.eval()

engine = ExamSynthesizer()
synth_bgr, gt_mask, target_bgr = engine.generate_sample()

# 預測
th, tw = 1024, 768
r_synth = cv2.resize(synth_bgr, (tw, th))
r_rgb = cv2.cvtColor(r_synth, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
norm_tensor = torch.from_numpy((r_rgb - mean) / std).permute(2, 0, 1).unsqueeze(0).float().to(device)

with torch.no_grad():
    pred_mask = torch.sigmoid(model(norm_tensor))[0, 0].cpu().numpy()

# 塗白修復
cleaned = r_synth.copy()
cleaned[pred_mask > 0.45] = [255, 255, 255]

# 視覺化
plt.figure(figsize=(18, 6))
plt.subplot(1, 4, 1)
plt.imshow(cv2.cvtColor(r_synth, cv2.COLOR_BGR2RGB))
plt.title('合成輸入 (含高密度手寫)')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(pred_mask, cmap='gray')
plt.title('預測筆跡 Mask')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(cv2.cvtColor(cleaned, cv2.COLOR_BGR2RGB))
plt.title('去手寫修復成果')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(cv2.cvtColor(cv2.resize(target_bgr, (tw, th)), cv2.COLOR_BGR2RGB))
plt.title('真實目標 (Ground Truth Clean)')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# 5. 下載訓練好的模型權重
from google.colab import files
files.download('weights/best_exam_remover.pth')
